# Demo 18 - MITRE ATT&CK coverage of the hunt library

**Pool:** Small · **Visual:** tactic x technique coverage heatmap

**The question:** where does our hunt library give us coverage, and where does it not?

Each hunt notebook is mapped to the MITRE ATT&CK tactics and techniques it exercises, and
the result is drawn as a grid. The filled cells are what you cover. The empty ones are the
point of the exercise.

KQL has no way to turn a hunt catalogue into a coverage picture, because the catalogue is
not data sitting in your workspace - it is a fact about your programme.

## 1. Connect to the data lake

`MicrosoftSentinelProvider` is the bridge between this notebook and your lake. The `spark`
session is handed to you by the Microsoft Sentinel kernel, so never create your own.

Nothing is read yet. This cell only opens the connection.

In [ ]:
from sentinel_lake.providers import MicrosoftSentinelProvider
from pyspark.sql import functions as F
data_provider = MicrosoftSentinelProvider(spark)

## 2. Map each hunt notebook to the ATT&CK techniques it exercises

`HUNT_MAP` is maintained by hand, and that is deliberate. It is a statement about what your
hunt library actually covers, not something to infer automatically.

MITRE ATT&CK organises adversary behaviour into **tactics** (the goal - what the attacker is
trying to achieve) and **techniques** (the method - how they go about it). Each hunt below
is tagged with both.

Add a row whenever you write a new hunt. The coverage picture is only ever as honest as
this dictionary.

In [ ]:
WORKSPACE = "your-workspace-name"

# hunt notebook -> list of (tactic, technique)
HUNT_MAP = {
    "demo03 beacon":        [("Command and Control","T1071 App Layer Protocol")],
    "demo07 impossible":    [("Initial Access","T1078 Valid Accounts")],
    "demo09 lateral graph": [("Lateral Movement","T1021 Remote Services")],
    "demo10 entropy":      [("Defense Evasion","T1027 Obfuscated Files")],
    "demo13 retro-hunt":   [("Command and Control","T1071 App Layer Protocol"),
                            ("Execution","T1204 User Execution")],
    "demo14 stack count":  [("Execution","T1059 Command/Script")],
    "demo15 first-seen":   [("Command and Control","T1071 App Layer Protocol")],
    "demo16 LOLBin":       [("Defense Evasion","T1218 System Binary Proxy Exec"),
                            ("Execution","T1059 Command/Script")],
    "demo17 timeline":     [("Discovery","T1057 Process Discovery")],
}

## 3. Build and draw the coverage matrix

Reshape the mapping into a grid of technique against tactic, count how many hunts touch
each cell, and render it as a heatmap.

**What to look for:** the empty cells. This chart exists to show you gaps, not to make you
feel good about the squares that are filled in. A tactic column with nothing in it is a
blind spot you have just written down.

In [ ]:
import pandas as pd, numpy as np, seaborn as sns, matplotlib.pyplot as plt

rows = []
for hunt, techs in HUNT_MAP.items():
    for tactic, tech in techs:
        rows.append({"tactic":tactic, "technique":tech, "hunt":hunt})
m = pd.DataFrame(rows)
cover = (m.groupby(["technique","tactic"]).size().reset_index(name="hunts")
           .pivot(index="technique", columns="tactic", values="hunts").fillna(0))

plt.figure(figsize=(11, 7))
sns.heatmap(cover, annot=True, cmap="YlGnBu", linewidths=.5, cbar_kws={"label":"# hunts"})
plt.title("Hunt-library coverage across MITRE ATT&CK tactics")
plt.tight_layout(); plt.show()
print("Techniques covered:", cover.shape[0])

## 4. Optionally weight a technique with live data

Coverage on paper is one thing. Whether the technique is actually being exercised in your
environment right now is another.

This cell counts recent executions of a few well-known proxy binaries as a live signal for
T1218, wrapped in a try/except so a missing table degrades to a message rather than ending
the run.

Extend the idea and the payoff is real: a coverage heatmap weighted by actual finding counts
tells you where to hunt *next*, rather than just where you have hunted before.

In [ ]:
# Optional live weighting: count today's LOLBin (T1218) hits to show a 'hot' technique
try:
    proc = data_provider.read_table("DeviceProcessEvents", WORKSPACE)
    n = (proc.filter(F.col("TimeGenerated") >= F.expr("current_timestamp() - INTERVAL 7 DAYS"))
             .filter(F.lower("FileName").isin(["certutil.exe","rundll32.exe","regsvr32.exe","mshta.exe"]))
             .count())
    print(f"Live signal - T1218 candidate executions (7d): {n}")
except Exception as e:
    print("(live weighting skipped:", e, ")")

## Why this is a notebook hunt, not a KQL query

Hunters live in ATT&CK. Building a coverage matrix from your hunt library and rendering it as a heatmap - then optionally weighting cells with live counts - is a pandas/Seaborn exercise. KQL has no way to turn your hunt catalogue into a coverage picture.